# Baseline Clause-Extraction Model (CUAD) — Training Notebook

This notebook fine-tunes a small model (`distilbert-base-uncased`) to find and highlight
4 clause types inside contracts:

- Governing Law
- Termination For Convenience
- Uncapped Liability
- Non-Compete

**How it works (simple version):** we treat this as a *question-answering* task. For every
contract, we ask a question like *"Highlight the parts related to Governing Law"*, and the
model learns to point to the exact piece of text that answers it — the same approach the
original CUAD paper used.

**How to use this notebook:**
1. Open this file in Google Colab (colab.research.google.com -> File -> Upload notebook).
2. Go to `Runtime > Change runtime type` and set **Hardware accelerator** to **GPU** (the free T4 is fine).
3. Go to `Runtime > Run all`.
4. Wait for training to finish — the cells run top to bottom automatically, no input needed.
5. At the very end, your browser will download a file called `clause_extraction_model.zip` —
   that is your trained model.

You do not need to upload any files yourself — the training data downloads automatically
inside this notebook.

In [ ]:
!pip install -q transformers datasets accelerate evaluate

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU is ON: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected!")
    print("Go to: Runtime > Change runtime type > Hardware accelerator > GPU, then run this cell again.")

## Step 1: Load the CUAD dataset

CUAD (Contract Understanding Atticus Dataset) has 510 real contracts with expert
annotations across 41 clause types. We only need 4 of them for this baseline, so the next
cells download the full dataset and keep only the rows that match our 4 categories.

In [ ]:
from huggingface_hub import hf_hub_download
import json

# The Hugging Face auto-loader for this dataset repo (load_dataset("theatticusproject/cuad"))
# is unreliable — depending on caching it can return raw PDF files instead of the actual
# clause annotations. So instead we download the official SQuAD-format annotation file
# directly from the repo and parse it ourselves. This is the same file the original CUAD
# paper released.
cuad_json_path = hf_hub_download(
    repo_id="theatticusproject/cuad",
    repo_type="dataset",
    filename="CUAD_v1/CUAD_v1.json",
)

with open(cuad_json_path, encoding="utf-8") as f:
    cuad_raw = json.load(f)

all_records = []
for document in cuad_raw["data"]:
    for paragraph in document["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            all_records.append(
                {
                    "id": qa["id"],
                    "question": qa["question"],
                    "context": context,
                    "answers": {
                        "answer_start": [a["answer_start"] for a in qa["answers"]],
                        "text": [a["text"] for a in qa["answers"]],
                    },
                }
            )

print(f"Total rows before filtering: {len(all_records)}")

In [ ]:
from datasets import Dataset
from collections import Counter

CATEGORIES = [
    "Governing Law",
    "Termination For Convenience",
    "Uncapped Liability",
    "Non-Compete",
]


def matched_category(question: str):
    # Match the exact quoted category name (e.g. `"Non-Compete"`), not a bare substring.
    # Some categories mention other category names inside their own description text
    # (e.g. "Competitive Restriction Exception" mentions "Non-Compete" in its details),
    # which would cause incorrect double-matches with a plain substring check.
    return next((c for c in CATEGORIES if f'"{c}"' in question), None)


filtered_records = []
for record in all_records:
    category = matched_category(record["question"])
    if category is not None:
        filtered_records.append({**record, "category": category})

filtered_dataset = Dataset.from_list(filtered_records)
print(f"Rows after filtering to our 4 categories: {len(filtered_dataset)}")
print(Counter(filtered_dataset["category"]))

# Quick peek at one example so you can see the shape of the data
print(filtered_dataset[0]["question"])
print(filtered_dataset[0]["answers"])

In [ ]:
split_dataset = filtered_dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print(f"Train examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")

## Step 2: Turn text into numbers (tokenization)

Models don't read words directly — everything gets converted into numbers first
("tokens"). This step also works out exactly which tokens correspond to the start and
end of each answer inside the contract, so the model has something concrete to learn
from. You don't need to change anything in this cell — just run it.

In [ ]:
from transformers import AutoTokenizer

MODEL_CHECKPOINT = "distilbert-base-uncased"
MAX_LENGTH = 384
STRIDE = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)


def preprocess(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]

        # No answer for this clause type in this contract -> point to [CLS] (index 0)
        if len(answer["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue

        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs


train_tokenized = train_dataset.map(
    preprocess, batched=True, remove_columns=train_dataset.column_names
)
val_tokenized = val_dataset.map(
    preprocess, batched=True, remove_columns=val_dataset.column_names
)

print(f"Tokenized train examples: {len(train_tokenized)}")
print(f"Tokenized validation examples: {len(val_tokenized)}")

## Step 3: Fine-tune the model

This is the actual training step. We start from `distilbert-base-uncased` (a small model
that already understands English well) and teach it, over 3 passes through the data
("epochs"), to point to the right clause text. On Colab's free GPU this usually takes
somewhere between a few minutes and about an hour, depending on how busy Colab is that day.

You'll see a progress bar and a "loss" number that should generally go down over time —
that means it's learning.

In [ ]:
from transformers import AutoModelForQuestionAnswering, Trainer, TrainingArguments

model = AutoModelForQuestionAnswering.from_pretrained(MODEL_CHECKPOINT)

training_args = TrainingArguments(
    output_dir="clause-extraction-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
)

trainer.train()

## Step 4: Sanity check — does it actually work?

Before trusting this model, let's manually look at a couple of predictions next to the
real, expert-labeled answer. This is exactly the kind of human review the project brief
calls for — never trust model output blindly, especially on legal documents.

In [ ]:
from transformers import pipeline

qa_pipeline = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

for i in range(3):
    example = val_dataset[i]
    prediction = qa_pipeline(question=example["question"], context=example["context"])
    print("=" * 80)
    print("Question:", example["question"])
    print("Predicted:", prediction["answer"], f"(confidence: {prediction['score']:.2f})")
    if example["answers"]["text"]:
        actual = example["answers"]["text"][0]
    else:
        actual = "(no clause of this type in this contract)"
    print("Actual:  ", actual)

## Step 5: Save and download your trained model

This packages the trained model into a single zip file and downloads it straight to your
computer — no Google Drive or Hugging Face account needed. Once it's downloaded, keep it
somewhere safe; a later step will load this model locally to actually run clause
extraction on real contracts.

In [ ]:
import shutil

SAVE_DIR = "clause-extraction-model-final"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

shutil.make_archive("clause_extraction_model", "zip", SAVE_DIR)
print("Saved and zipped to clause_extraction_model.zip")

from google.colab import files
files.download("clause_extraction_model.zip")